# SE Capstone: Stream + Task Pipeline & Dynamic Table

## Requirement
- **Pipeline A**: Stream + Task to detect new ADS-B data and transform into a conformed model within 15-minute SLA
- **Pipeline B**: Dynamic Table approach for comparison
- Both join ADS-B data with FAA reference data
- Only **common fields** between kapa-0001 (newer firmware) and kbfi-0001 (older firmware)

## Schema Differences
| Field | kapa-0001 (flat JSON) | kbfi-0001 (nested JSON) |
|-|-|-|
| ICAO Hex | `hexid` | `hex` (inside `aircraft[]`) |
| Callsign | `ident` | `flight` (inside `aircraft[]`) |
| Latitude | `lat` | `lat` (inside `aircraft[]`) |
| Longitude | `lon` | `lon` (inside `aircraft[]`) |
| Baro Altitude | `baro_alt` | `alt_baro` (inside `aircraft[]`) |
| GPS Altitude | `gps_alt` | `alt_geom` (inside `aircraft[]`) |
| Ground Speed | `gs` | `gs` (inside `aircraft[]`) |
| Heading | `heading` | `track` (inside `aircraft[]`) |
| Squawk | `squawk` | `squawk` (inside `aircraft[]`) |
| Timestamp | `clock` (epoch) | `now` (parent-level epoch) |

In [ ]:
-- CONTEXT SETUP
USE ROLE CAPSTONE26_RPEGU;
USE DATABASE CAPSTONE26_DB;
USE WAREHOUSE CAP26_WH;

## Step 1: Create the Conformed Model Target Table
Contains only **common elements** between both firmware versions plus:
- `SOURCE_FILENAME` and `SOURCE_ROW_NUM` for traceability/audit
- `STATION_ID` to identify the firmware source
- FAA enrichment columns from the join

In [ ]:
-- Target table for Stream + Task pipeline
CREATE TABLE IF NOT EXISTS CAPSTONE26_DB.STAGING.ADSB_CONFORMED_VIA_STREAM (
    ICAO_HEX           VARCHAR     COMMENT 'ICAO 24-bit hex address',
    CALLSIGN           VARCHAR     COMMENT 'Flight callsign / tail number',
    LATITUDE           FLOAT       COMMENT 'Aircraft latitude',
    LONGITUDE          FLOAT       COMMENT 'Aircraft longitude',
    ALTITUDE_BARO      VARCHAR     COMMENT 'Barometric altitude (can be ground)',
    ALTITUDE_GEO       VARCHAR     COMMENT 'GPS/geometric altitude',
    GROUND_SPEED       FLOAT       COMMENT 'Ground speed in knots',
    HEADING            FLOAT       COMMENT 'Track/heading in degrees',
    SQUAWK             VARCHAR     COMMENT 'Transponder squawk code',
    EVENT_TIME         TIMESTAMP   COMMENT 'UTC timestamp of the position report',
    STATION_ID         VARCHAR     COMMENT 'Source station: kapa-0001 or kbfi-0001',
    SOURCE_FILENAME    VARCHAR     COMMENT 'Original source file path',
    SOURCE_ROW_NUM     NUMBER      COMMENT 'Row number in source file',
    FAA_N_NUMBER       VARCHAR     COMMENT 'FAA registration N-number',
    FAA_NAME           VARCHAR     COMMENT 'Registered owner name',
    FAA_MFR_MDL_CODE   VARCHAR     COMMENT 'Manufacturer model code',
    FAA_TYPE_AIRCRAFT  VARCHAR     COMMENT 'Aircraft type code',
    LOAD_TIMESTAMP     TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
)
COMMENT = 'Conformed ADS-B + FAA model populated by Stream + Task pipeline';

## Step 2: Create Normalization Views
These views handle the different JSON structures:
- **kapa-0001**: Flat JSON, one record per row
- **kbfi-0001**: Nested JSON with `aircraft[]` array that must be FLATTENed

In [ ]:
-- Normalize kapa-0001 (flat JSON)
CREATE OR REPLACE VIEW CAPSTONE26_DB.STAGING.V_ADSB_KAPA_NORMALIZED AS
SELECT
    UPPER(RAW_DATA:hexid::VARCHAR)          AS ICAO_HEX,
    TRIM(RAW_DATA:ident::VARCHAR)            AS CALLSIGN,
    RAW_DATA:lat::FLOAT                      AS LATITUDE,
    RAW_DATA:lon::FLOAT                      AS LONGITUDE,
    RAW_DATA:baro_alt::VARCHAR               AS ALTITUDE_BARO,
    RAW_DATA:gps_alt::VARCHAR                AS ALTITUDE_GEO,
    RAW_DATA:gs::FLOAT                       AS GROUND_SPEED,
    RAW_DATA:heading::FLOAT                  AS HEADING,
    RAW_DATA:squawk::VARCHAR                 AS SQUAWK,
    TO_TIMESTAMP(RAW_DATA:clock::NUMBER)     AS EVENT_TIME,
    'kapa-0001'                              AS STATION_ID,
    SOURCE_FILENAME,
    SOURCE_ROW_NUM
FROM CAPSTONE26_DB.RAW.ADSB_RAW_KAPA
WHERE RAW_DATA:lat IS NOT NULL
  AND RAW_DATA:lon IS NOT NULL;

In [ ]:
-- Normalize kbfi-0001 (nested JSON: LATERAL FLATTEN on aircraft[])
CREATE OR REPLACE VIEW CAPSTONE26_DB.STAGING.V_ADSB_KBFI_NORMALIZED AS
SELECT
    UPPER(ac.value:hex::VARCHAR)             AS ICAO_HEX,
    TRIM(ac.value:flight::VARCHAR)           AS CALLSIGN,
    ac.value:lat::FLOAT                      AS LATITUDE,
    ac.value:lon::FLOAT                      AS LONGITUDE,
    ac.value:alt_baro::VARCHAR               AS ALTITUDE_BARO,
    ac.value:alt_geom::VARCHAR               AS ALTITUDE_GEO,
    ac.value:gs::FLOAT                       AS GROUND_SPEED,
    ac.value:track::FLOAT                    AS HEADING,
    ac.value:squawk::VARCHAR                 AS SQUAWK,
    TO_TIMESTAMP(RAW_DATA:now::NUMBER)       AS EVENT_TIME,
    'kbfi-0001'                              AS STATION_ID,
    SOURCE_FILENAME,
    SOURCE_ROW_NUM
FROM CAPSTONE26_DB.RAW.ADSB_RAW_KBFI,
LATERAL FLATTEN(input => RAW_DATA:aircraft) ac
WHERE ac.value:lat IS NOT NULL
  AND ac.value:lon IS NOT NULL;

In [ ]:
-- Combined view with FAA enrichment
CREATE OR REPLACE VIEW CAPSTONE26_DB.STAGING.V_ADSB_CONFORMED AS
SELECT
    a.*, 
    f.N_NUMBER AS FAA_N_NUMBER, f.NAME AS FAA_NAME,
    f.MFR_MDL_CODE AS FAA_MFR_MDL_CODE, f.TYPE_AIRCRAFT AS FAA_TYPE_AIRCRAFT
FROM (
    SELECT * FROM CAPSTONE26_DB.STAGING.V_ADSB_KAPA_NORMALIZED
    UNION ALL
    SELECT * FROM CAPSTONE26_DB.STAGING.V_ADSB_KBFI_NORMALIZED
) a
LEFT JOIN CAPSTONE26_DB.RAW.FAA_MASTER f
    ON UPPER(TRIM(f.MODE_S_CODE_HEX)) = a.ICAO_HEX;

In [ ]:
-- Quick preview
SELECT * FROM CAPSTONE26_DB.STAGING.V_ADSB_CONFORMED LIMIT 20;

## Step 3: Pipeline A - Stream + Task

### Architecture:
```
ADSB_RAW_KAPA --> STREAM_KAPA --+
                                +--> TASK (every 5 min) --> ADSB_CONFORMED_VIA_STREAM
ADSB_RAW_KBFI --> STREAM_KBFI --+         + FAA JOIN
```
- APPEND_ONLY streams detect new inserts only
- Task runs every 5 min, checks WHEN streams have data
- Meets the 15-minute SLA

In [ ]:
-- 3a. Create APPEND_ONLY streams on both raw tables
CREATE OR REPLACE STREAM CAPSTONE26_DB.RAW.STREAM_ADSB_KAPA
    ON TABLE CAPSTONE26_DB.RAW.ADSB_RAW_KAPA
    APPEND_ONLY = TRUE
    COMMENT = 'Detects new kapa-0001 records for Stream+Task pipeline';

CREATE OR REPLACE STREAM CAPSTONE26_DB.RAW.STREAM_ADSB_KBFI
    ON TABLE CAPSTONE26_DB.RAW.ADSB_RAW_KBFI
    APPEND_ONLY = TRUE
    COMMENT = 'Detects new kbfi-0001 records for Stream+Task pipeline';

In [ ]:
-- 3b. Create the Task (runs every 5 min, only when streams have data)
CREATE OR REPLACE TASK CAPSTONE26_DB.STAGING.TASK_ADSB_CONFORMED
    WAREHOUSE = CAP26_WH
    SCHEDULE = '5 MINUTE'
    COMMENT = 'Transforms new ADS-B data into conformed model with FAA enrichment'
    WHEN
        SYSTEM$STREAM_HAS_DATA('CAPSTONE26_DB.RAW.STREAM_ADSB_KAPA')
        OR SYSTEM$STREAM_HAS_DATA('CAPSTONE26_DB.RAW.STREAM_ADSB_KBFI')
AS
BEGIN
    -- Insert new kapa-0001 records (flat JSON)
    INSERT INTO CAPSTONE26_DB.STAGING.ADSB_CONFORMED_VIA_STREAM
        (ICAO_HEX, CALLSIGN, LATITUDE, LONGITUDE, ALTITUDE_BARO, ALTITUDE_GEO,
         GROUND_SPEED, HEADING, SQUAWK, EVENT_TIME, STATION_ID,
         SOURCE_FILENAME, SOURCE_ROW_NUM,
         FAA_N_NUMBER, FAA_NAME, FAA_MFR_MDL_CODE, FAA_TYPE_AIRCRAFT)
    SELECT
        UPPER(s.RAW_DATA:hexid::VARCHAR),
        TRIM(s.RAW_DATA:ident::VARCHAR),
        s.RAW_DATA:lat::FLOAT,
        s.RAW_DATA:lon::FLOAT,
        s.RAW_DATA:baro_alt::VARCHAR,
        s.RAW_DATA:gps_alt::VARCHAR,
        s.RAW_DATA:gs::FLOAT,
        s.RAW_DATA:heading::FLOAT,
        s.RAW_DATA:squawk::VARCHAR,
        TO_TIMESTAMP(s.RAW_DATA:clock::NUMBER),
        'kapa-0001',
        s.SOURCE_FILENAME,
        s.SOURCE_ROW_NUM,
        f.N_NUMBER, f.NAME, f.MFR_MDL_CODE, f.TYPE_AIRCRAFT
    FROM CAPSTONE26_DB.RAW.STREAM_ADSB_KAPA s
    LEFT JOIN CAPSTONE26_DB.RAW.FAA_MASTER f
        ON UPPER(TRIM(f.MODE_S_CODE_HEX)) = UPPER(s.RAW_DATA:hexid::VARCHAR)
    WHERE s.RAW_DATA:lat IS NOT NULL AND s.RAW_DATA:lon IS NOT NULL;

    -- Insert new kbfi-0001 records (nested JSON, needs FLATTEN)
    INSERT INTO CAPSTONE26_DB.STAGING.ADSB_CONFORMED_VIA_STREAM
        (ICAO_HEX, CALLSIGN, LATITUDE, LONGITUDE, ALTITUDE_BARO, ALTITUDE_GEO,
         GROUND_SPEED, HEADING, SQUAWK, EVENT_TIME, STATION_ID,
         SOURCE_FILENAME, SOURCE_ROW_NUM,
         FAA_N_NUMBER, FAA_NAME, FAA_MFR_MDL_CODE, FAA_TYPE_AIRCRAFT)
    SELECT
        UPPER(ac.value:hex::VARCHAR),
        TRIM(ac.value:flight::VARCHAR),
        ac.value:lat::FLOAT,
        ac.value:lon::FLOAT,
        ac.value:alt_baro::VARCHAR,
        ac.value:alt_geom::VARCHAR,
        ac.value:gs::FLOAT,
        ac.value:track::FLOAT,
        ac.value:squawk::VARCHAR,
        TO_TIMESTAMP(s.RAW_DATA:now::NUMBER),
        'kbfi-0001',
        s.SOURCE_FILENAME,
        s.SOURCE_ROW_NUM,
        f.N_NUMBER, f.NAME, f.MFR_MDL_CODE, f.TYPE_AIRCRAFT
    FROM CAPSTONE26_DB.RAW.STREAM_ADSB_KBFI s,
    LATERAL FLATTEN(input => s.RAW_DATA:aircraft) ac
    LEFT JOIN CAPSTONE26_DB.RAW.FAA_MASTER f
        ON UPPER(TRIM(f.MODE_S_CODE_HEX)) = UPPER(ac.value:hex::VARCHAR)
    WHERE ac.value:lat IS NOT NULL AND ac.value:lon IS NOT NULL;
END;

In [ ]:
-- 3c. Resume the task (created in SUSPENDED state)
ALTER TASK CAPSTONE26_DB.STAGING.TASK_ADSB_CONFORMED RESUME;

In [ ]:
-- 3d. Monitor task
SELECT *
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    TASK_NAME => 'TASK_ADSB_CONFORMED',
    SCHEDULED_TIME_RANGE_START => DATEADD('hour', -24, CURRENT_TIMESTAMP())
))
ORDER BY SCHEDULED_TIME DESC
LIMIT 20;

In [ ]:
-- Stream status
SELECT SYSTEM$STREAM_HAS_DATA('CAPSTONE26_DB.RAW.STREAM_ADSB_KAPA') AS kapa_has_data,
       SYSTEM$STREAM_HAS_DATA('CAPSTONE26_DB.RAW.STREAM_ADSB_KBFI') AS kbfi_has_data;

## Step 4: Pipeline B - Dynamic Table

### Architecture:
```
ADSB_RAW_KAPA --+
                +--> DYNAMIC TABLE (auto-refresh, TARGET_LAG = 10 min)
ADSB_RAW_KBFI --+        + FAA JOIN
```
- No Stream or Task needed
- Snowflake auto-manages incremental refresh
- Single DDL statement defines the entire pipeline

In [ ]:
-- Dynamic Table with 10-minute target lag (within 15-min SLA)
CREATE OR REPLACE DYNAMIC TABLE CAPSTONE26_DB.STAGING.ADSB_CONFORMED_VIA_DT
    TARGET_LAG = '10 MINUTES'
    WAREHOUSE = CAP26_WH
    COMMENT = 'Conformed ADS-B + FAA model via Dynamic Table (auto-refresh)'
AS
SELECT
    a.ICAO_HEX, a.CALLSIGN, a.LATITUDE, a.LONGITUDE,
    a.ALTITUDE_BARO, a.ALTITUDE_GEO, a.GROUND_SPEED, a.HEADING,
    a.SQUAWK, a.EVENT_TIME, a.STATION_ID,
    a.SOURCE_FILENAME, a.SOURCE_ROW_NUM,
    f.N_NUMBER AS FAA_N_NUMBER, f.NAME AS FAA_NAME,
    f.MFR_MDL_CODE AS FAA_MFR_MDL_CODE, f.TYPE_AIRCRAFT AS FAA_TYPE_AIRCRAFT
FROM (
    -- kapa-0001: flat JSON
    SELECT
        UPPER(RAW_DATA:hexid::VARCHAR) AS ICAO_HEX,
        TRIM(RAW_DATA:ident::VARCHAR) AS CALLSIGN,
        RAW_DATA:lat::FLOAT AS LATITUDE, RAW_DATA:lon::FLOAT AS LONGITUDE,
        RAW_DATA:baro_alt::VARCHAR AS ALTITUDE_BARO,
        RAW_DATA:gps_alt::VARCHAR AS ALTITUDE_GEO,
        RAW_DATA:gs::FLOAT AS GROUND_SPEED,
        RAW_DATA:heading::FLOAT AS HEADING,
        RAW_DATA:squawk::VARCHAR AS SQUAWK,
        TO_TIMESTAMP(RAW_DATA:clock::NUMBER) AS EVENT_TIME,
        'kapa-0001' AS STATION_ID,
        SOURCE_FILENAME, SOURCE_ROW_NUM
    FROM CAPSTONE26_DB.RAW.ADSB_RAW_KAPA
    WHERE RAW_DATA:lat IS NOT NULL AND RAW_DATA:lon IS NOT NULL
    UNION ALL
    -- kbfi-0001: nested JSON (FLATTEN aircraft[])
    SELECT
        UPPER(ac.value:hex::VARCHAR) AS ICAO_HEX,
        TRIM(ac.value:flight::VARCHAR) AS CALLSIGN,
        ac.value:lat::FLOAT AS LATITUDE, ac.value:lon::FLOAT AS LONGITUDE,
        ac.value:alt_baro::VARCHAR AS ALTITUDE_BARO,
        ac.value:alt_geom::VARCHAR AS ALTITUDE_GEO,
        ac.value:gs::FLOAT AS GROUND_SPEED,
        ac.value:track::FLOAT AS HEADING,
        ac.value:squawk::VARCHAR AS SQUAWK,
        TO_TIMESTAMP(r.RAW_DATA:now::NUMBER) AS EVENT_TIME,
        'kbfi-0001' AS STATION_ID,
        r.SOURCE_FILENAME, r.SOURCE_ROW_NUM
    FROM CAPSTONE26_DB.RAW.ADSB_RAW_KBFI r,
    LATERAL FLATTEN(input => r.RAW_DATA:aircraft) ac
    WHERE ac.value:lat IS NOT NULL AND ac.value:lon IS NOT NULL
) a
LEFT JOIN CAPSTONE26_DB.RAW.FAA_MASTER f
    ON UPPER(TRIM(f.MODE_S_CODE_HEX)) = a.ICAO_HEX;

In [ ]:
-- Monitor Dynamic Table refresh
SELECT *
FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(
    NAME => 'CAPSTONE26_DB.STAGING.ADSB_CONFORMED_VIA_DT'
))
ORDER BY REFRESH_END_TIME DESC
LIMIT 10;

## Step 5: Validate and Compare

In [ ]:
-- Row count comparison
SELECT 'Stream+Task' AS pipeline, COUNT(*) AS row_count FROM CAPSTONE26_DB.STAGING.ADSB_CONFORMED_VIA_STREAM
UNION ALL
SELECT 'Dynamic Table', COUNT(*) FROM CAPSTONE26_DB.STAGING.ADSB_CONFORMED_VIA_DT;

In [ ]:
-- Station breakdown
SELECT STATION_ID, COUNT(*) AS rows, COUNT(DISTINCT ICAO_HEX) AS unique_aircraft
FROM CAPSTONE26_DB.STAGING.ADSB_CONFORMED_VIA_DT
GROUP BY STATION_ID;

In [ ]:
-- FAA enrichment match rate
SELECT
    COUNT(*) AS total_rows,
    COUNT(FAA_N_NUMBER) AS faa_matched,
    ROUND(COUNT(FAA_N_NUMBER) * 100.0 / NULLIF(COUNT(*),0), 2) AS match_pct
FROM CAPSTONE26_DB.STAGING.ADSB_CONFORMED_VIA_DT;

In [ ]:
-- Audit: trace a record to its source file
SELECT ICAO_HEX, CALLSIGN, EVENT_TIME, SOURCE_FILENAME, SOURCE_ROW_NUM
FROM CAPSTONE26_DB.STAGING.ADSB_CONFORMED_VIA_DT
WHERE CALLSIGN IS NOT NULL
LIMIT 10;

## POC ReadOut: Comparison Summary

| Aspect | Stream + Task | Dynamic Table |
|-|-|-|
| **Setup complexity** | More objects (stream, task, INSERT logic) | Single DDL statement |
| **Refresh control** | CRON/schedule + WHEN condition | TARGET_LAG (Snowflake manages) |
| **SLA guarantee** | Task schedule (5 min) + processing | TARGET_LAG = 10 min |
| **Compute** | Warehouse runs only when stream has data | Snowflake auto-manages refreshes |
| **Incremental** | Yes (stream tracks new rows only) | Yes (auto-detected) |
| **Customization** | Full control over INSERT logic, error handling | Limited to single SELECT |
| **Monitoring** | TASK_HISTORY, manual alerts | DYNAMIC_TABLE_REFRESH_HISTORY |
| **Best for** | Complex multi-step transformations | Simple declarative pipelines |

In [ ]:
-- CLEANUP: Suspend when not testing (saves credits)
-- ALTER TASK CAPSTONE26_DB.STAGING.TASK_ADSB_CONFORMED SUSPEND;
-- ALTER DYNAMIC TABLE CAPSTONE26_DB.STAGING.ADSB_CONFORMED_VIA_DT SUSPEND;